Pre-analysis of the data

In [3]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Extract CSV's to PD's dataframes
df_financial = pd.read_csv("data/raw/financial_impact.csv")
df_incidents = pd.read_csv("data/raw/incidents_master.csv")
df_market = pd.read_csv("data/raw/market_impact.csv")

We do need to do a general overview of the dataset we are working with. Since we are going to work as it was a real-world scenario, we will analyse the common columns on each of the csv.

In [5]:
# GENERAL VIEW
# row count, column count, join keys, date ranges, data source types

def general_view(df, df_others = []):
    print("GENERAL VIEW")
    df.info()
    print("==========")
    print("OTHER DATAFRAMES COMMON COLUMNS")

    if len(df_others) == 0:
        return
    
    relations_list = []
    global_common = []
    for col in df.columns:
        for i, df_object in enumerate(df_others):
            if len(relations_list) <= i:
                relations_list.append([i])
            
            if col in df_object.columns:
                relations_list[i].append(col)

    global_common = relations_list[0][1:]

    for df_object in relations_list:
        global_common = set(global_common).intersection(set(df_object[1:]))
        print(f"- Dataframe {df_object[0]}:", df_object[1:])
    
    print("Global common columns:", set(global_common))

general_view(df_incidents, [df_financial, df_market])

GENERAL VIEW
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 850 entries, 0 to 849
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   incident_id               850 non-null    object 
 1   company_name              850 non-null    object 
 2   company_revenue_usd       850 non-null    float64
 3   country_hq                850 non-null    object 
 4   industry_primary          850 non-null    object 
 5   industry_secondary        153 non-null    object 
 6   employee_count            850 non-null    int64  
 7   is_public_company         850 non-null    bool   
 8   stock_ticker              412 non-null    object 
 9   incident_date             850 non-null    object 
 10  incident_date_estimated   850 non-null    bool   
 11  discovery_date            850 non-null    object 
 12  disclosure_date           850 non-null    object 
 13  attack_vector_primary     850 non-null    object 
 1

We just identified a common identifier for all the dataframes, the column "incident_id". Now we would be able of trying to cross the data in case we need to for the common entries.

Now we will manage one of the most important problems in the pre-analysis stage. We need to get rid of the missing data, based on the their specific case.

In [13]:
incidents_nulls = df_incidents.isnull().mean().sort_values(ascending=False)
financial_nulls = df_financial.isnull().mean().sort_values(ascending=False)
market_nulls = df_market.isnull().mean().sort_values(ascending=False)

incidents_nulls[incidents_nulls > 0], financial_nulls[financial_nulls > 0], market_nulls[market_nulls > 0]


(review_flag                 0.917647
 industry_secondary          0.820000
 attack_vector_secondary     0.751765
 notes                       0.748235
 data_source_secondary       0.545882
 stock_ticker                0.515294
 downtime_hours              0.505882
 attributed_group            0.432941
 attribution_confidence      0.432941
 attack_chain                0.323529
 data_compromised_records    0.291765
 data_type                   0.291765
 dtype: float64,
 ransom_paid_usd         0.889460
 ransom_source           0.889460
 regulatory_fine_usd     0.830334
 ransom_demanded_usd     0.735219
 notes                   0.681234
 insurance_payout_usd    0.440874
 dtype: float64,
 notes                     0.743017
 days_to_price_recovery    0.100559
 dtype: float64)

As you can see, there are some columns with a high missingness-ratio. Even though, all the ones that could be considered as unrealable data (>60% missing), are under a regular collection ratio for its type of data. Some examples are high missingness on ransomwere related fields, since only ones that could fill that information are ransomware attack cases, 

** Explanation on how and why we are merging

df_merged is our master

In [15]:
df_merged = df_incidents.merge(df_financial, on='incident_id', how='inner').merge(df_market, on='incident_id', how='inner')
df_merged.shape

(329, 80)

In [19]:
# Split database into numerical and categorical subsets
num_cols = df_merged.select_dtypes(include=['number']).columns
cat_cols = df_merged.select_dtypes(include=['object', 'string', 'category', 'bool']).columns
df_numerical = df_merged[num_cols] 
df_categorical = df_merged[cat_cols]

In [ ]:
df_numerical.describe() # If df is mixed num+cat, it shows analysis only on numerical features either way

,company_revenue_usd,employee_count,data_compromised_records,downtime_hours,confidence_tier,quality_score,direct_loss_usd,ransom_demanded_usd,ransom_paid_usd,recovery_cost_usd,...,car_0_to_90,t_statistic_1d,p_value_1d,t_statistic_30d,p_value_30d,market_cap_at_disclosure,volume_ratio_disclosure,pre_incident_volatility_30d,post_incident_volatility_30d,days_to_price_recovery
count,3.290000e+02,3.290000e+02,2.310000e+02,155.000000,329.000000,329.000000,3.290000e+02,8.400000e+01,4.000000e+01,3.290000e+02,...,329.000000,329.000000,329.000000,329.000000,329.000000,3.290000e+02,329.000000,329.000000,329.000000,294.000000
mean,1.550088e+10,7.748355e+04,3.579353e+06,99.471871,2.133739,79.555471,4.238009e+07,7.496491e+06,3.116261e+06,3.270513e+07,...,-0.012173,-2.613695,0.661158,-0.983797,0.890514,7.824025e+10,2.747007,0.023750,0.037178,109.272109
std,2.473178e+10,1.466648e+05,3.660443e+07,163.859142,1.199740,12.508881,1.130260e+08,1.321549e+07,6.655387e+06,9.937133e+07,...,0.021202,2.213962,0.401353,1.679624,0.255675,1.368705e+11,0.727687,0.008575,0.015034,102.374109
min,3.560745e+07,6.700000e+01,1.000000e+03,4.000000,1.000000,51.750000,9.913191e+04,5.000000e+04,2.816497e+04,7.137618e+04,...,-0.095579,-15.612200,0.000200,-10.336500,0.000200,9.267550e+07,1.508100,0.010135,0.011879,5.000000
25%,9.018864e+08,3.539000e+03,1.222700e+04,23.680000,1.000000,71.430000,3.600000e+06,7.864379e+05,3.816215e+05,2.305462e+06,...,-0.024803,-3.462600,0.268700,-1.812400,1.000000,3.186086e+09,2.112800,0.016327,0.024743,28.000000
50%,5.217619e+09,2.025100e+04,5.456900e+04,51.650000,2.000000,79.850000,1.018142e+07,2.348851e+06,1.071370e+06,7.702210e+06,...,-0.010017,-2.168900,0.915550,-0.841000,1.000000,1.920208e+10,2.750300,0.023417,0.034481,57.000000
75%,1.917962e+10,9.739000e+04,3.081495e+05,114.520000,3.000000,89.900000,3.189020e+07,6.511687e+06,3.260266e+06,2.304697e+07,...,0.003644,-1.139800,1.000000,0.047500,1.000000,9.165653e+10,3.381200,0.030778,0.047886,177.000000
max,1.488980e+11,1.136850e+06,5.497365e+08,1198.930000,4.000000,99.780000,1.065993e+09,7.500000e+07,4.075677e+07,1.238193e+09,...,0.038986,1.185600,1.000000,4.671000,1.000000,1.059593e+12,3.993300,0.039934,0.077072,363.000000
